# Inventory Analysis Capstone

Start by importing all necessary Python libraries

In [3]:
import pandas as pd
import requests
import yfinance
import psycopg2
import sqlalchemy


### Data Acquisition

Read in the dataset with the 500 companies first

In [4]:
companies = pd.read_csv('data/SP500.csv')
companies.head(5)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373


Collect Net Inventory from SEC filings for each company

In [69]:
def fetch_sec_data(cik):
    headers = {'User-Agent' : 'marshalln@etown.edu', 'Host': 'data.sec.gov'}
    cik = str(cik).zfill(10)
    url = 'https://data.sec.gov/api/xbrl/companyfacts/CIK' + cik + '.json'
    response = requests.get(url,headers=headers)
    response.raise_for_status()
    try:
        response = response.json()['facts']['us-gaap']['InventoryNet']['units']['USD']
    except:
        print(cik)
        response = None
    return response


In [ ]:
data = pd.DataFrame(columns=['company', 'industry', 'inventory', 'gross_profit', 'revenue'])
for i in range(len(companies)):
    symbol = companies['Symbol'][i]
    sec_data = fetch_sec_data(companies['CIK'][i])
    if sec_data is not None:
        data.loc[len(data)] = [symbol, companies['GICS Sub-Industry'][i], sec_data]

data.head()



0
1
2
3
4


,company,industry,data
0,MMM,Industrial Conglomerates,"[{'end': '2008-12-31', 'val': 3013000000, 'acc..."
1,AOS,Building Products,"[{'end': '2009-12-31', 'val': 215100000, 'accn..."
2,ABT,Health Care Equipment,"[{'end': '2007-12-31', 'val': 2951442000, 'acc..."
3,ABBV,Biotechnology,"[{'end': '2012-12-31', 'val': 1091000000, 'acc..."


Export to a csv file for easy use later

In [ ]:
data.to_csv('inventory_data.csv', index=False)

### Data Cleaning

Turn Collected SEC Data into Cleaner Format